### Demo sample images
- Saves test images for the app demo into app/samples
- One correctly predicted test image per class (10 in total)
- One known failure: a Late_blight leaf the model calls healthy
- All are test images, never used in training
- Saved as PNG so the pixels are not changed by compression

In [1]:
import os
os.chdir("/Users/eloelo/Projects/plant-disease-detection")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from PIL import Image

test_data = tf.data.Dataset.load("data/splits_tfds/test")

def resize_image(image, label):
    image = tf.image.resize(image, (224, 224))
    return image, label

test_data = test_data.map(resize_image).batch(32)

short_names = [
    "bacterial_spot",
    "early_blight",
    "healthy",
    "late_blight",
    "leaf_mold",
    "septoria_leaf_spot",
    "spider_mites",
    "target_spot",
    "tomato_mosaic_virus",
    "yellow_leaf_curl_virus",
]

y_true = []
for images, labels in test_data:
    for label in labels.numpy():
        y_true.append(int(label))
y_true = np.array(y_true)

print("Test images:", len(y_true))

2026-09-29 00:08:33.860687: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-09-29 00:08:33.860703: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-09-29 00:08:33.860715: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
I0000 00:00:1790620713.860725  992409 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1790620713.860740  992409 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Test images: 1825


2026-09-29 00:08:34.078192: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [2]:
model = keras.models.load_model("models/mobilenet_finetuned.keras")

probabilities = model.predict(test_data, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

print("Test accuracy:", round(np.mean(y_pred == y_true) * 100, 2), "%")

2026-09-29 00:08:47.231211: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Test accuracy: 96.82 %


In [3]:
os.makedirs("app/samples", exist_ok=True)

saved = {}
failure_saved = False
index = 0

for images, labels in test_data:
    for image in images.numpy():
        true_class = y_true[index]
        pred_class = y_pred[index]
        leaf = Image.fromarray(image.astype("uint8"))

        if pred_class == true_class and true_class not in saved:
            path = "app/samples/correct_" + short_names[true_class] + ".png"
            leaf.save(path)
            saved[true_class] = path

        if true_class == 3 and pred_class == 2 and not failure_saved:
            leaf.save("app/samples/failure_late_blight_called_healthy.png")
            failure_saved = True

        index += 1

print("Correct samples saved:", len(saved))
print("Failure case saved:", failure_saved)

Correct samples saved: 10
Failure case saved: True


2026-09-29 00:09:12.694089: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
val_data = tf.data.Dataset.load("data/splits_tfds/val")
val_data = val_data.map(resize_image).batch(32)

val_true = []
for images, labels in val_data:
    for label in labels.numpy():
        val_true.append(int(label))
val_true = np.array(val_true)

val_probabilities = model.predict(val_data, verbose=0)
val_pred = np.argmax(val_probabilities, axis=1)
val_confidence = np.max(val_probabilities, axis=1)

for level in [0.5, 0.6, 0.7, 0.8, 0.9]:
    low = val_confidence < level
    high = val_confidence >= level

    low_count = int(np.sum(low))
    low_accuracy = np.mean(val_pred[low] == val_true[low]) if low_count > 0 else 0
    high_accuracy = np.mean(val_pred[high] == val_true[high])

    print("Level", level, "| below:", low_count, "images, accuracy", round(low_accuracy * 100, 1), "% | above: accuracy", round(high_accuracy * 100, 1), "%")

Level 0.5 | below: 11 images, accuracy 45.5 % | above: accuracy 96.8 %
Level 0.6 | below: 42 images, accuracy 61.9 % | above: accuracy 97.3 %
Level 0.7 | below: 84 images, accuracy 61.9 % | above: accuracy 98.2 %
Level 0.8 | below: 120 images, accuracy 67.5 % | above: accuracy 98.6 %
Level 0.9 | below: 179 images, accuracy 72.6 % | above: accuracy 99.1 %


### Choosing the low-confidence warning level
- Checked on the validation set (not test), since choosing the level is a decision
- Below 70% confidence: 84 of 1,812 images (4.6%), only 61.9% correct
- Above 70%: 98.2% correct
- 60% and 70% give the same accuracy below the level (61.9%), but 70% catches twice as many mistakes (about 32 vs 16)
- Above 70%, the extra warned images are more and more correct, so the warning would lose meaning
- Chosen level: 70%; it also catches the known failure case (Healthy at 66.5%)